In [9]:
import pandas as pd
import numpy as np
import os
from DaySim import DaysimSummary
from Survey import DaysimSummary_Survey

In [10]:
DAMPING_FACTOR = 0.85
THRESHOLD = 1 # ALLOWABLE PERCENTAGE POINT DIFFERENCE
COPY_BACK = False

In [11]:
processed_dir = "data/processed"
interim_dir = "data/interim"
#coefficient_files_input_dir = "Z:/projects/CHCNGA/US0049529.0645/DaySim_update/ChattaModel/1_Model-Files/5_DaySim/Inputs/9_Coefficients"
coefficient_files_input_dir = "9_Coefficients" # temp
coefficient_files_output_dir = interim_dir # replace with coefficient_files_input_dir 

In [12]:
MAPPING_FILE = "calibration_mapping.csv"
LOG_FILE = "calibration_log.txt"
F12_FILES = {
    file.split('Coefficients_Chattanooga')[0]: file 
    for file in os.listdir(coefficient_files_input_dir) 
    if file.endswith('.F12')
}


In [13]:
F12_FILES

{'AutoOwnership': 'AutoOwnershipCoefficients_Chattanooga.F12',
 'EscortTourMode': 'EscortTourModeCoefficients_Chattanooga.F12',
 'IndividualPersonDayPattern': 'IndividualPersonDayPatternCoefficients_Chattanooga.F12',
 'IntermediateStopGeneration': 'IntermediateStopGenerationCoefficients_Chattanooga.F12',
 'IntermediateStopLocation': 'IntermediateStopLocationCoefficients_Chattanooga.F12',
 'OtherHomeBasedTourMode': 'OtherHomeBasedTourModeCoefficients_Chattanooga_tnc.F12',
 'OtherHomeBasedTourTime': 'OtherHomeBasedTourTimeCoefficients_Chattanooga.F12',
 'OtherTourDestination': 'OtherTourDestinationCoefficients_Chattanooga.F12',
 'PayToParkAtWorkplace': 'PayToParkAtWorkplaceCoefficients_Chattanooga.F12',
 'PersonExactNumberOfTours': 'PersonExactNumberOfToursCoefficients_Chattanooga.F12',
 'SchoolLocation': 'SchoolLocationCoefficients_Chattanooga.F12',
 'SchoolTourMode': 'SchoolTourModeCoefficients_Chattanooga.F12',
 'SchoolTourTime': 'SchoolTourTimeCoefficients_Chattanooga.F12',
 'Transit

In [14]:
# Models to calibrate

MODELS ={
    'AutoOwnership': True,
    'WorkLocation': False,
    'SchoolLocation': False,
    'DayPatternPurpose': False,
    'DayPatternCount': False,

}

# Prepare coefficient files 

In [15]:
import re

def _parse_f12_lines(path):
    """Parse F12 file; return list of dicts with parsed data or raw lines."""
    records = []
    with open(path, 'r') as fh:
        for raw in fh:
            tokens = raw.split()
            if len(tokens) >= 4:
                try:
                    records.append({
                        'is_data': True,
                        'end': int(tokens[0]),
                        'name': tokens[1],
                        'constr': tokens[2],
                        'beta': float(tokens[3]),
                        'stderr': float(tokens[4]) if len(tokens) > 4 else 0.0,
                        'raw': raw,
                    })
                except ValueError:
                    records.append({'is_data': False, 'raw': raw})
            else:
                records.append({'is_data': False, 'raw': raw})
    return records
   

def read_f12(model_name):
    """
    Read an F12 file from F12_DIR and return a DataFrame with columns:
        end, name, constr, beta, stderr
    One row per data line; non-data lines (headers, blanks) are excluded.
    Display this DataFrame to inspect the file contents.
    """
    path = os.path.join(coefficient_files_input_dir , F12_FILES[model_name])
    records = _parse_f12_lines(path)
    df = pd.DataFrame(
        [{'end': r['end'], 'name': r['name'], 'constr': r['constr'],
          'beta': r['beta'], 'stderr': r['stderr']}
         for r in records if r['is_data']]
    )
    return df

def _replace_at_position(raw, char_start, char_end, new_str):
    return raw[:char_start] + new_str + raw[char_end:]

def write_f12(model_name, end_to_beta):
    """
    Write an updated copy of the F12 file to F12_OUT_DIR.
    Reads from F12_DIR, replaces Beta at its exact character position for each
    END in end_to_beta, preserves all other content unchanged.
    """
    src_path = os.path.join(coefficient_files_input_dir ,     F12_FILES[model_name])
    dst_path = os.path.join(coefficient_files_output_dir , F12_FILES[model_name])
    os.makedirs(coefficient_files_output_dir , exist_ok=True)

    records   = _parse_f12_lines(src_path)
    out_lines = []
    n_updated = 0

    for r in records:
        if r['is_data'] and r['end'] in end_to_beta:
            new_beta               = end_to_beta[r['end']]
            beta_start, beta_end   = r['positions'][3]
            orig_width             = beta_end - beta_start
            new_str                = f'{new_beta:.6f}'
            if len(new_str) < orig_width:
                new_str = new_str.rjust(orig_width)
            out_lines.append(_replace_at_position(r['raw'], beta_start, beta_end, new_str))
            n_updated += 1
        else:
            out_lines.append(r['raw'])

    with open(dst_path, 'w') as fh:
        fh.writelines(out_lines)
    print(f'  {F12_FILES[model_name]}: {n_updated} Beta(s) updated  ->  {dst_path}')




# Run Summaries

In [8]:
#model  = DaysimSummary()
survey = DaysimSummary_Survey()

runVehAvailability = True, loading data...
runWrkSchLocationChoice = True, loading data...
runTripMode = True, loading data...
runTourMode = True, loading data...
runTripDestination = True, loading data...
runTourDestination = True, loading data...
runTripTOD = True, loading data...
runTourTOD = True, loading data...
runDayPattern = True, loading data...


# Mapping targets to coefficients 

# Calibration